In [5]:
import os
import re
import time
from pathlib import Path
from datetime import datetime

import joblib
import mlflow
import mlflow.xgboost
import xgboost as xgb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

from sklearn.model_selection import train_test_split
from mlflow.models import infer_signature


mlflow.set_tracking_uri("http://localhost:5000")

EXPERIMENT_NAME = "xgb_Airports_Experiments"
REGISTERED_MODEL_NAME = "AIRPORTS_REGRESSION_Predictor"

mlflow.set_experiment(EXPERIMENT_NAME)


OUTPUT_DIR = Path("models")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("MLflow version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())



def get_next_experiment_version(experiment_name):
    experiment = mlflow.get_experiment_by_name(experiment_name)

    if experiment is None:
        experiment_id = mlflow.create_experiment(experiment_name)
        return 1

    runs = mlflow.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["attributes.start_time DESC"]
    )

    if runs.empty:
        return 1

    versions = []

    for run_name in runs.get(
        "tags.mlflow.runName", pd.Series(dtype=str)
    ).dropna():
        match = re.search(r"_v(\d+)_", run_name)
        if match:
            versions.append(int(match.group(1)))

    return max(versions, default=0) + 1




train_csv = Path("datasets\cleaned_airports.csv")

if not train_csv.exists():
    raise FileNotFoundError(
        f"Cannot find {train_csv.resolve()}. "
        "Put the CSV in the notebook's working directory "
        "or update train_csv to its actual path."
    )

df = pd.read_csv(train_csv)

if "FLT_TOT_1" not in df.columns:
    raise ValueError(
        "The training CSV must contain a 'label' column."
    )

X = df.drop(columns=["FLT_TOT_1"])
y = df["FLT_TOT_1"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    # stratify=y
)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_test))
print("Number of features:", X_train.shape[1])



xgb_params = {
    'objective': 'reg:squarederror',
    'n_estimators': 3250,
    'learning_rate': 0.09345130495773961,
    'max_depth': 8,
    'subsample': 0.9848348894944928,
    'colsample_bytree': 0.6627564120404217,
    'min_child_weight': 9.439574346445202,
    'gamma': 0.38102029226599476,
    'reg_alpha': 0.0059313599732688845,
    'reg_lambda': 6.313235266795839,
    'early_stopping_rounds': 50,
    'random_state': 42
}


@mlflow.trace
def predict_flights(model, features):
    """Run predictions and record an MLflow trace."""
    return model.predict(features)


version = get_next_experiment_version(EXPERIMENT_NAME)

run_name = (
    f"{EXPERIMENT_NAME}_v{version}_"                        ## xgb_flights_Prediction_Experiments_v1
    f"{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
)

start_time = time.time()

with mlflow.start_run(run_name=run_name) as run:

    print("Run ID:", run.info.run_id)
    print("Experiment version:", version)

    # Record useful metadata
    mlflow.set_tags({
        "model_family": "XGBoost",
        "task": "regression",
        "dataset": "Airport info",
        "experiment_version": str(version)
    })

    # Log dataset location and schema
    mlflow.log_param("dataset_path", str(train_csv.resolve()))
    mlflow.log_param("dataset_rows", len(df))
    mlflow.log_param("number_of_features", X.shape[1])

    dataset = mlflow.data.from_pandas(
        df,
        source=str(train_csv.resolve()),
        targets="FLT_TOT_1"
    )

    mlflow.log_input(dataset, context="training")

    # Log model parameters
    mlflow.log_params(xgb_params)

    # Train the model
    xgb_model = xgb.XGBRegressor(**xgb_params)
    xgb_model.fit(X_train,
                  y_train,
                  eval_set=[(X_test, y_test)]
                 )

    # Predict and evaluate
    xgb_pred = predict_flights(xgb_model, X_test)

    mae = mean_absolute_error(y_test, xgb_pred)
    rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
    r2 = r2_score(y_test, xgb_pred)

    # accuracy = accuracy_score(y_test, xgb_pred)
    # report = classification_report(
    #     y_test,
    #     xgb_pred,
    #     output_dict=True,
    #     zero_division=0
    # )

    # Log metrics
    mlflow.log_metric("Root Mean Squared Error", float(rmse))
    mlflow.log_metric("R2 Score", float(r2))
    mlflow.log_metric(
        "model_size_mb",
        float(
            len(joblib.dumps(xgb_model)) / (1024 * 1024)
        ) if hasattr(joblib, "dumps") else 0.0
    )

    # # Log selected classification metrics
    # for average in ["macro avg", "weighted avg"]:
    #     mlflow.log_metric(
    #         f"f1_{average.replace(' ', '_')}",
    #         float(report[average]["f1-score"])
    #     )

    print(f"R2 Score: {r2}")

    pkl_path = OUTPUT_DIR / f"xgb_model_airport_{version}.pkl"

    joblib.dump(xgb_model, pkl_path)

    # Log the file directly in Run Artifacts
    mlflow.log_artifact(
        str(pkl_path),
        artifact_path="joblib_model"
    )

    mlflow.log_metric(
        "pkl_size_mb",
        pkl_path.stat().st_size / (1024 * 1024)
    )


    input_example = X_test.iloc[:5].copy()
    example_predictions = xgb_model.predict(input_example)

    signature = infer_signature(
        input_example,
        example_predictions
    )

    model_info = mlflow.xgboost.log_model(
        xgb_model,
        name="xgboost-model",
        registered_model_name=REGISTERED_MODEL_NAME,
        input_example=input_example,
        signature=signature,
        pip_requirements=[
            f"mlflow=={mlflow.__version__}",
            f"xgboost=={xgb.__version__}",
            f"pandas=={pd.__version__}",
            "numpy"
        ]
    )

    print("Logged model URI:", model_info.model_uri)
    print("Logged model ID:", model_info.model_id)


    local_model_dir = (
        OUTPUT_DIR / f"mlflow_xgboost_model_v{version}"
    )

    mlflow.xgboost.save_model(
        path=str(local_model_dir),
        xgb_model=xgb_model,
        input_example=input_example,
        signature=signature,
        pip_requirements=[
            f"mlflow=={mlflow.__version__}",
            f"xgboost=={xgb.__version__}",
            f"pandas=={pd.__version__}",
            "numpy"
        ]
    )


    # python_env.yaml, conda.yaml, and the model file.
    print("Local MLflow model files:")
    for file_path in sorted(local_model_dir.rglob("*")):
        if file_path.is_file():
            print(" -", file_path.relative_to(local_model_dir))

    # Upload the complete directory into the Run's Artifacts.
    mlflow.log_artifacts(
        str(local_model_dir),
        artifact_path="xgboost_model_files"
    )

    feature_importance_df = pd.DataFrame({
        "Feature": X_train.columns.astype(str),
        "Importance": xgb_model.feature_importances_
    }).sort_values(
        by="Importance",
        ascending=False
    )

    plt.figure(figsize=(10, 6))

    sns.barplot(
        data=feature_importance_df.head(20),
        x="Importance",
        y="Feature"
    )

    plt.title("XGBoost Feature Importance")
    plt.tight_layout()

    plot_path = OUTPUT_DIR / f"xgb_feature_importance_v{version}.png"
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.close()

    mlflow.log_artifact(
        str(plot_path),
        artifact_path="plots"
    )

    # Log a text classification report
    report_path = OUTPUT_DIR / f"classification_report_v{version}.txt"

    # with open(report_path, "w", encoding="utf-8") as f:
    #     f.write(
    #         classification_report(
    #             y_test,
    #             xgb_pred,
    #             zero_division=0
    #         )
    #     )

    # mlflow.log_artifact(
    #     str(report_path),
    #     artifact_path="evaluation"
    # )

    duration = time.time() - start_time
    mlflow.log_metric("training_and_logging_seconds", duration)

    print(f"Run completed in {duration:.2f} seconds.")
    print("Run ID:", run.info.run_id)
    print("Registered model:", REGISTERED_MODEL_NAME)
    print("Model URI:", model_info.model_uri)

print("Finished successfully.")


MLflow version: 3.17.2
Tracking URI: http://localhost:5000
Training rows: 703004
Validation rows: 175752
Number of features: 9
Run ID: 16f6981759d34fc48d0a79cd8e5e31ae
Experiment version: 5


f:\Programs\anaconda3\Lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
f:\Programs\anaconda3\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/model

[0]	validation_0-rmse:235.90127
[1]	validation_0-rmse:223.63738
[2]	validation_0-rmse:215.01219
[3]	validation_0-rmse:206.25668
[4]	validation_0-rmse:196.14869
[5]	validation_0-rmse:187.63879
[6]	validation_0-rmse:177.47145
[7]	validation_0-rmse:172.21188
[8]	validation_0-rmse:165.80553
[9]	validation_0-rmse:158.84850
[10]	validation_0-rmse:154.51693
[11]	validation_0-rmse:152.95991
[12]	validation_0-rmse:149.53689
[13]	validation_0-rmse:141.94196
[14]	validation_0-rmse:135.47192
[15]	validation_0-rmse:129.83886
[16]	validation_0-rmse:126.67789
[17]	validation_0-rmse:122.22844
[18]	validation_0-rmse:119.02209
[19]	validation_0-rmse:114.06145
[20]	validation_0-rmse:112.18971
[21]	validation_0-rmse:108.81325
[22]	validation_0-rmse:104.80900
[23]	validation_0-rmse:104.05724
[24]	validation_0-rmse:101.40435
[25]	validation_0-rmse:99.98626
[26]	validation_0-rmse:98.33143
[27]	validation_0-rmse:95.59592
[28]	validation_0-rmse:91.83918
[29]	validation_0-rmse:90.20685
[30]	validation_0-rmse:88

f:\Programs\anaconda3\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Registered model 'AIRPORTS_REGRESSION_Predictor' already exists. Creating a new version of this model...
2026/11/02 21:14:58 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: AIRPORTS_REGRESSION_Predictor, version 5
Created version '5' of model 'AIRPORTS_REGRESSION_Predictor'.


Logged model URI: models:/m-8591b5fd018e4ac38896eb077e1da93b
Logged model ID: m-8591b5fd018e4ac38896eb077e1da93b
Local MLflow model files:
 - conda.yaml
 - input_example.json
 - MLmodel
 - model.ubj
 - python_env.yaml
 - requirements.txt
 - serving_input_example.json
Run completed in 245.74 seconds.
Run ID: 16f6981759d34fc48d0a79cd8e5e31ae
Registered model: AIRPORTS_REGRESSION_Predictor
Model URI: models:/m-8591b5fd018e4ac38896eb077e1da93b
🏃 View run xgb_Airports_Experiments_v5_2026-11-02_21-10-54 at: http://localhost:5000/#/experiments/2/runs/16f6981759d34fc48d0a79cd8e5e31ae
🧪 View experiment at: http://localhost:5000/#/experiments/2
Finished successfully.


Trace(trace_id='tr-41afd3e6e37e9f05aab337b0349a6cce')